In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import torch
import matplotlib.pyplot as plt

## Dataset — California Housing

Using the [California Housing dataset](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset) from scikit-learn, derived from the **1990 California census**.

**What we're predicting:** median house value for a block group (a small geographic area of ~600–3000 people), in units of $100,000. So a target of `2.5` means the median house in that block costs $250,000.

**The 8 input features:**

| Feature | What it means |
|---|---|
| `MedInc` | Median income of households in the block (in tens of thousands of dollars) |
| `HouseAge` | Median age of houses in the block |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Total population of the block |
| `AveOccup` | Average number of people per household |
| `Latitude` | Geographic latitude of the block |
| `Longitude` | Geographic longitude of the block |

**Size:** 20,640 rows — each row is one block group in California.

This is a good dataset for multivariate regression because the features have very different scales (income vs. population vs. lat/lon), which makes it a realistic test of whether the model can handle real-world messy data.

In [ ]:
data = fetch_california_housing()
X, Y = data.data, data.target

# split into training and test sets — the model only learns from x_train/y_train
# x_test/y_test are set aside to check how well it does on data it's never seen
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(x_train[0])

In [ ]:
X_train = torch.tensor(x_train, dtype=torch.float32)
X_test  = torch.tensor(x_test, dtype=torch.float32)
Y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
Y_test  = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# the 8 features are on very different scales — income is 0-15, population is in thousands, etc.
# if we don't normalize, the gradient updates will be all over the place and training breaks
# so we shift each feature to have mean=0 and std=1
mean = X_train.mean(dim=0)
std  = X_train.std(dim=0)
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std  # use train stats here, not test — test data should stay unseen

In [ ]:
class LinearRegressionModel(torch.nn.Module):

    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        # 8 inputs (one per feature), 1 output (the predicted house price)
        self.linear = torch.nn.Linear(8, 1)

    def forward(self, x):
        return self.linear(x)

In [ ]:
multivariate_regression = LinearRegressionModel()
criterion = torch.nn.MSELoss()  # measures how far off our predictions are
optimizer = torch.optim.SGD(multivariate_regression.parameters(), lr=0.01)  # adjusts weights each step

In [ ]:
losses = []  # track loss over time so we can plot it later

for epoch in range(500):
    pred_y = multivariate_regression(X_train)  # make a prediction
    loss = criterion(pred_y, Y_train)           # see how wrong it is

    optimizer.zero_grad()  # clear gradients from last step (they accumulate otherwise)
    loss.backward()        # figure out which direction to adjust each weight
    optimizer.step()       # take a step in that direction

    losses.append(loss.item())

    if epoch % 50 == 0:
        print(f'epoch {epoch}, loss {loss.item()}')

In [ ]:
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss')
plt.show()

In [ ]:
# check how the model does on data it never trained on
# if test loss is close to train loss, the model actually learned the pattern
# if test loss is much higher, it just memorized the training data
with torch.no_grad():
    test_pred = multivariate_regression(X_test)
    test_loss = criterion(test_pred, Y_test)

print(f'train loss: {losses[-1]:.4f}')
print(f'test loss:  {test_loss.item():.4f}')